# BERTopic Parameter Tuning

Individual question-by-question BERTopic fitting with isolated parameter tuning.

Each question is fit separately with the same baseline parameters from the main `fit_bertopic()` function. This allows for targeted tuning of each question's topic model.

In [14]:
# Imports
import pandas as pd
import numpy as np

# BERTopic stack
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
import hdbscan
from umap import UMAP
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech


# Optional: Ollama/LLM support (uncomment if using)
import openai
from bertopic.representation import OpenAI

# Configure OpenAI for topic representation
client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key='ollama'
)

representation_model_llm = OpenAI(client, 
                              model='mistral')

In [2]:
# Load data
df = pd.read_csv("../data/student-responses.csv")

# Filter only Bar Chart experiment responses
df = df[df["experiment"] == "Bar chart"]

print(f"Total responses: {len(df)}")
print(f"Dataframe shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Total responses: 654
Dataframe shape: (654, 21)
Columns: ['id', 'section_sis_id', 'section', 'attempt', 'What components of the experiment are clearer now than they were as a participant What questions do you still have for the experimenter Write 3 5 sentences reflecting on the abstract', 'Paste the response code you received after participating in the graphics experiment here', 'As of today I am at least 19 years of age', 'My instructor may share my reflection responses with the researchers in this study', 'What do you think the purpose of the experiment was', 'What elements of experimental design such as randomization or the use of a control group do you think were present in the experiment Why', 'What hypotheses might the experimenter have been testing', 'What sources of error are involved in this experiment', 'What variables were examined For each variable identify whether it was quantitative or categorical', 'In this class you ll be learning about the process of scientific investi

## Baseline BERTopic Parameters

All questions will be fit using these shared parameters. Modify here to tune globally, or override in individual question cells for targeted tuning.

In [3]:
# Initialize baseline models and configurations

# Embedding model
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# UMAP parameters
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# HDBSCAN parameters
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=1,
    cluster_selection_method="eom",
    prediction_data=True
)

# Vectorizer model
vectorizer_model = CountVectorizer(
    stop_words="english",
    min_df=2,
    max_df=0.95
)

# C-TF-IDF model
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

# Representation model (KeyBERT by default, change to OpenAI if using Ollama)
representation_model = KeyBERTInspired()

print("Baseline parameters initialized.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5866.08it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline parameters initialized.


# Global Settings

Each question will be structued as follows:

1. Fit a generic BERTopic
2. Update the model with different parameters UMAP and HDBSCAN

# Per-Question BERTopic Fitting

## Pre-Experiment

### Q1: In this class, you'll be learning about the process of scientific investigation...

**Column Index:** 13

In [12]:
# Q1: Extract and clean responses
docs_q1 = df.iloc[:, 13].dropna().astype(str).str.strip()
docs_q1 = docs_q1[docs_q1.ne("")].tolist()

# Fit Initial BERTopic
model_q1 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.9),
        "POS": PartOfSpeech("en_core_web_sm")
        #"LLM": OpenAI(client, model='mistral', prompt=prompt)
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=10,
        n_components=10,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True #Required for probabilities
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q1, probs_q1 = model_q1.fit_transform(docs_q1)
model_q1.get_topic_info()

2026-04-13 23:19:14,057 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6519.66it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 20/20 [00:06<00:00,  3.06it/s]
2026-04-13 23:19:24,246 - BERTopic - Embedding - Completed ✓
2026-04-13 23:19:24,246 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-13 23:19:25,224 - BERTopic - Dimensionality - Completed ✓
2026-04-13 23:19:25,224 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 23:19:25,244 - BERTopic - Cluster - Completed ✓
2026-04-13 23:19:25,245 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,172,-1_class_actually_ones_learning,"[class, actually, ones, learning, results thin...","[investigation perspective, public scientific,...","[class, actually, lots, data analyzing, comple...","[class, ones, learning, depth, lots, simple, a...",[From the perspective of a researcher I know t...
1,0,87,0_depth_researcher lot_think researcher_compar...,"[depth, researcher lot, think researcher, comp...","[public researchers, researchers perspective, ...","[depth, compared general, trying, parts, think...","[depth, parts, good, article, simple, final, f...","[Generally speaking, I believe that the perspe..."
2,1,49,1_individual_times_thought_importance,"[individual, times, thought, importance, easy,...","[investigation perspective, think researchers,...","[individual, importance, errors, hard, idea, d...","[individual, times, importance, easy, errors, ...","[Prior to this class, I was pretty familiar wi..."
3,2,45,2_contrast_peer review_news articles_ongoing,"[contrast, peer review, news articles, ongoing...","[understanding scientific, research process, p...","[contrast, peer review, journey, collect analy...","[contrast, ongoing, systematic, complexities, ...",[The process of scientific investigation invol...
4,3,42,3_news_public science_big_discoveries,"[news, public science, big, discoveries, trial...","[public science, view science, consuming scien...","[public science, big, figuring, step process, ...","[news, public science, big, discoveries, trial...","[From the perspective of a researcher, the sci..."
5,4,40,4_peers_factor_conducted_field,"[peers, factor, conducted, field, actual, effo...","[investigation researchers, researchers perspe...","[peers, factor, conducted, actual, thought, ge...","[peers, factor, field, actual, good, effort, t...",[I think that research can take on many differ...
6,5,40,5_theory_reject_disprove_theories,"[theory, reject, disprove, theories, curiosity...","[believe scientific, using scientific, science...","[reject, individuals, causes, recording, creat...","[theory, theories, curiosity, solution, indivi...",[I think that science happens through the basi...
7,6,38,6_look like_stats_experts_investigation look,"[look like, stats, experts, investigation look...","[researches, think researchers, researchers po...","[look like, numbers, researches, drawn, true, ...","[stats, experts, numbers, researches, factual,...",[From the perspective of a researcher learning...
8,7,38,7_group_researched_gathered_tested,"[group, researched, gathered, tested, answered...","[research process, hypothesis research, proces...","[answered, necessary, plan, doesn, population,...","[group, plan, necessary, population, control, ...",[I think that the first part of the experiment...
9,8,29,8_interested_attention_regarding_results science,"[interested, attention, regarding, results sci...","[conduct research, researchers point, differen...","[interested, researchers point, life, pay, acc...","[interested, attention, eyes, ones, life, diff...",[I feel the processes are both different. Bein...


In [13]:
model_q1.visualize_documents(docs_q1, hide_annotations=True)

In [18]:
model_q1.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,172,-1_Scientific understanding gap (general publi...,[Scientific understanding gap (general public ...,"[investigation perspective, public scientific,...","[class, actually, lots, data analyzing, comple...","[class, ones, learning, depth, lots, simple, a...",[From the perspective of a researcher I know t...
1,0,87,0_Topic: Comparison of Researcher and Public P...,[Topic: Comparison of Researcher and Public Pe...,"[public researchers, researchers perspective, ...","[depth, compared general, trying, parts, think...","[depth, parts, good, article, simple, final, f...","[Generally speaking, I believe that the perspe..."
2,1,49,1_Topic: Scientific Investigation Process Pers...,[Topic: Scientific Investigation Process Persp...,"[investigation perspective, think researchers,...","[individual, importance, errors, hard, idea, d...","[individual, times, importance, easy, errors, ...","[Prior to this class, I was pretty familiar wi..."
3,2,45,2_Topic: Scientific process perception gap,[Topic: Scientific process perception gap],"[understanding scientific, research process, p...","[contrast, peer review, journey, collect analy...","[contrast, ongoing, systematic, complexities, ...",[The process of scientific investigation invol...
4,3,42,3_Science perceptions between researchers and ...,[Science perceptions between researchers and p...,"[public science, view science, consuming scien...","[public science, big, figuring, step process, ...","[news, public science, big, discoveries, trial...","[From the perspective of a researcher, the sci..."
5,4,40,4_Topic: Consumer vs. Researcher perspectives ...,[Topic: Consumer vs. Researcher perspectives i...,"[investigation researchers, researchers perspe...","[peers, factor, conducted, actual, thought, ge...","[peers, factor, field, actual, good, effort, t...",[I think that research can take on many differ...
6,5,40,5_Science process,[Science process],"[believe scientific, using scientific, science...","[reject, individuals, causes, recording, creat...","[theory, theories, curiosity, solution, indivi...",[I think that science happens through the basi...
7,6,38,6_Scientific investigation process,[Scientific investigation process],"[researches, think researchers, researchers po...","[look like, numbers, researches, drawn, true, ...","[stats, experts, numbers, researches, factual,...",[From the perspective of a researcher learning...
8,7,38,7_Scientific Research Process,[Scientific Research Process],"[research process, hypothesis research, proces...","[answered, necessary, plan, doesn, population,...","[group, plan, necessary, population, control, ...",[I think that the first part of the experiment...
9,8,29,8_Researcher vs Consumer Perspective,[Researcher vs Consumer Perspective],"[conduct research, researchers point, differen...","[interested, researchers point, life, pay, acc...","[interested, attention, eyes, ones, life, diff...",[I feel the processes are both different. Bein...


## Post-Experiment

### Q2: What do you think the purpose of the experiment was?

**Column Index:** 8

In [27]:
# Q2: Extract and clean responses
docs_q2 = df.iloc[:, 8].dropna().astype(str).str.strip()
docs_q2 = docs_q2[docs_q2.ne("")].tolist()

print(f"Q2 - Total responses: {len(docs_q2)}")

# Fit BERTopic
model_q2 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm")
        #"LLM": OpenAI(client, model='mistral', prompt=prompt)
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=10,
        n_components=10,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=20,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True #Required for probabilities
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q2, probs_q2 = model_q2.fit_transform(docs_q2)
model_q2.get_topic_info()

Q2 - Total responses: 521


2026-04-13 23:36:07,410 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6376.81it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:02<00:00,  7.52it/s]
2026-04-13 23:36:13,587 - BERTopic - Embedding - Completed ✓
2026-04-13 23:36:13,587 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-13 23:36:14,331 - BERTopic - Dimensionality - Completed ✓
2026-04-13 23:36:14,332 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 23:36:14,346 - BERTopic - Cluster - Completed ✓
2026-04-13 23:36:14,347 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,47,-1_blocks_block_perspective_measure,"[blocks, block, perspective, measure, certain,...","[depth perception, people visualize, blocks, e...","[blocks, perspective, measure, perceive things...","[blocks, block, perspective, certain, tall, co...",[I was a little confused in the beginning of t...
1,0,144,0_hypothesis_taking_population_people read,"[hypothesis, taking, population, people read, ...","[experiment students, experiment understand, e...","[hypothesis, people read, questions, experimen...","[hypothesis, population, questions, question, ...",[Figuring out our problem solving skills. Mayb...
2,1,135,1_analyze_3d graphs_read_looking,"[analyze, 3d graphs, read, looking, people est...","[read graphs, graphs purpose, understand stati...","[analyze, 3d graphs, graphs easier, bar graphs...","[online, bars, reading, ways, screen, way, typ...",[The project aimed to investigate how students...
3,2,68,2_object_shapes_objects_things,"[object, shapes, objects, things, 3d models, l...","[experiment difference, experiment test, exper...","[shapes, objects, 3d models, larger, depth per...","[object, shapes, objects, things, percentage, ...",[I think the purpose of the experiment was to ...
4,3,53,3_bar graphs_bar graph_bars_values,"[bar graphs, bar graph, bars, values, heights,...","[bar graphs, bar graph, difference bars, size ...","[bar graphs, bar graph, graphs think, 3d bar, ...","[bars, values, heights, bar, perspectives, com...",[I think the purpose of the experiment was to ...
5,4,28,4_images_graphs 3d_experiment different_2d graphs,"[images, graphs 3d, experiment different, 2d g...","[experiment understand, experiment difference,...","[graphs 3d, 2d graphs, visualization, affect p...","[images, visualization, different ways, beginn...","[At the beginning of the experiment, I was com..."
6,5,26,5_3d printed_printed_charts_digital 3d,"[3d printed, printed, charts, digital 3d, 2d d...","[3d charts, 3d graphs, charts purpose, 3d bar,...","[3d printed, charts, 3d digital, 3d charts, ba...","[charts, perceptual, visuals, researchers, typ...",[This study aimed to:\n\n*\n\nReplicate and ex...
7,6,20,6_experiment likely_people perception_context_...,"[experiment likely, people perception, context...","[depth perception, perception size, people per...","[experiment likely, people perception, 3d mode...","[context, objects, proportion, relative, answe...",[The purpose of this experiment is to see how ...


In [28]:
model_q2.visualize_documents(docs_q2, hide_annotations=True)

### Q3: What hypotheses might the experimenter have been testing?

**Column Index:** 10

In [48]:
# Q3: Extract and clean responses
docs_q3 = df.iloc[:, 10].dropna().astype(str).str.strip()
docs_q3 = docs_q3[docs_q3.ne("")].tolist()

print(f"Q3 - Total responses: {len(docs_q3)}")

# Fit BERTopic
model_q3 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm")
        #"LLM": OpenAI(client, model='mistral', prompt=prompt)
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=18,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True #Required for probabilities
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q3, probs_q3 = model_q3.fit_transform(docs_q3)
model_q3.get_topic_info()

Q3 - Total responses: 517


2026-04-13 23:53:23,885 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6171.10it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:03<00:00,  5.53it/s]
2026-04-13 23:53:33,317 - BERTopic - Embedding - Completed ✓
2026-04-13 23:53:33,318 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-13 23:53:34,158 - BERTopic - Dimensionality - Completed ✓
2026-04-13 23:53:34,161 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 23:53:34,182 - BERTopic - Cluster - Completed ✓
2026-04-13 23:53:34,182 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,135,-1_color_bar graphs_colors_bigger,"[color, bar graphs, colors, bigger, reading, 5...","[bar graphs, bar graph, interpret data, graphs...","[color, bar graphs, bigger, reading, values, s...","[color, colors, bigger, values, stats, questio...",[They might have hyponthesized that participan...
1,0,151,0_experiment_based_shapes_estimations,"[experiment, based, shapes, estimations, relat...","[hypothesis students, hypotheses experimenter,...","[shapes, experiment testing, sizes, object, gr...","[experiment, shapes, estimations, relative, th...",[The experiment might have been testing whethe...
2,1,149,1_charts_digital_printed_3d printed,"[charts, digital, printed, 3d printed, physica...","[compared 3d, 3d graphs, bar graphs, graphs 3d...","[charts, 3d printed, 3d graphs, 2d graphs, per...","[charts, digital, physical, chart, screen, per...",[The experimenters likely had several hypothes...
3,2,29,2_bars_bar graph_tall_estimation,"[bars, bar graph, tall, estimation, closer, he...","[bars compared, bar height, size bar, bar grap...","[bar graph, estimation, heights bar, size diff...","[bars, tall, closer, estimation, angles, apart...",[If smaller bars in a bar graph are next to la...
4,3,22,3_statistics_taking_class_stats,"[statistics, taking, class, stats, numerical, ...","[statistic, hypothesis students, statistics, s...","[statistics, statistic, unl students, students...","[statistics, class, stats, numerical, statisti...",[STAT 218 students will guess the correct pres...
5,4,16,4_null hypothesis_alternative_alternative hypo...,"[null hypothesis, alternative, alternative hyp...","[null hypothesis, testing hypothesis, alternat...","[null hypothesis, graphs 3d, difference accura...","[null hypothesis, alternative, alternative hyp...",[The null hypothesis the experimenter may have...
6,5,15,5_blocks_taller_believe_think experimenter,"[blocks, taller, believe, think experimenter, ...","[testing students, students accurately, blocks...","[think experimenter, estimate proportions, ski...","[blocks, taller, skills, degree, shapes, lengt...",[Students are better able to estimate proporti...


In [49]:
model_q3.visualize_documents(docs_q3, hide_annotations=True)

### Q4: What sources of error are involved in this experiment?

**Column Index:** 11

In [ ]:
# Q4: Extract and clean responses
docs_q4 = df.iloc[:, 11].dropna().astype(str).str.strip()
docs_q4 = docs_q4[docs_q4.ne("")].tolist()

print(f"Q4 - Total responses: {len(docs_q4)}")

# Fit BERTopic
model_q4 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)

topics_q4, probs_q4 = model_q4.fit_transform(docs_q4)
print(f"Q4 - Topics found: {len(set(topics_q4)) - 1}")

### Q5: What variables were examined? For each variable, identify whether it was quantitative or categorical.

**Column Index:** 12

In [ ]:
# Q5: Extract and clean responses
docs_q5 = df.iloc[:, 12].dropna().astype(str).str.strip()
docs_q5 = docs_q5[docs_q5.ne("")].tolist()

print(f"Q5 - Total responses: {len(docs_q5)}")

# Fit BERTopic
model_q5 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)

topics_q5, probs_q5 = model_q5.fit_transform(docs_q5)
print(f"Q5 - Topics found: {len(set(topics_q5)) - 1}")

### Q6: What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why?

**Column Index:** 9

In [ ]:
# Q6: Extract and clean responses
docs_q6 = df.iloc[:, 9].dropna().astype(str).str.strip()
docs_q6 = docs_q6[docs_q6.ne("")].tolist()

print(f"Q6 - Total responses: {len(docs_q6)}")

# Fit BERTopic
model_q6 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)

topics_q6, probs_q6 = model_q6.fit_transform(docs_q6)
print(f"Q6 - Topics found: {len(set(topics_q6)) - 1}")

## Abstract Reflection

### Q7: What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter?

**Column Index:** 4

In [ ]:
# Q7: Extract and clean responses
docs_q7 = df.iloc[:, 4].dropna().astype(str).str.strip()
docs_q7 = docs_q7[docs_q7.ne("")].tolist()

print(f"Q7 - Total responses: {len(docs_q7)}")

# Fit BERTopic
model_q7 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)

topics_q7, probs_q7 = model_q7.fit_transform(docs_q7)
print(f"Q7 - Topics found: {len(set(topics_q7)) - 1}")

## Presentation Reflection

### Q8: How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ?

**Column Index:** 14

In [ ]:
# Q8: Extract and clean responses
docs_q8 = df.iloc[:, 14].dropna().astype(str).str.strip()
docs_q8 = docs_q8[docs_q8.ne("")].tolist()

print(f"Q8 - Total responses: {len(docs_q8)}")

# Fit BERTopic
model_q8 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)

topics_q8, probs_q8 = model_q8.fit_transform(docs_q8)
print(f"Q8 - Topics found: {len(set(topics_q8)) - 1}")

### Q9: What components were emphasized in the presentation that weren't emphasized in the abstract? Why do you think that is?

**Column Index:** 16

In [ ]:
# Q9: Extract and clean responses
docs_q9 = df.iloc[:, 16].dropna().astype(str).str.strip()
docs_q9 = docs_q9[docs_q9.ne("")].tolist()

print(f"Q9 - Total responses: {len(docs_q9)}")

# Fit BERTopic
model_q9 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)

topics_q9, probs_q9 = model_q9.fit_transform(docs_q9)
print(f"Q9 - Topics found: {len(set(topics_q9)) - 1}")

### Q10: What critiques do you have of this study and its design? What would have made the study better?

**Column Index:** 17

In [ ]:
# Q10: Extract and clean responses
docs_q10 = df.iloc[:, 17].dropna().astype(str).str.strip()
docs_q10 = docs_q10[docs_q10.ne("")].tolist()

print(f"Q10 - Total responses: {len(docs_q10)}")

# Fit BERTopic
model_q10 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)

topics_q10, probs_q10 = model_q10.fit_transform(docs_q10)
print(f"Q10 - Topics found: {len(set(topics_q10)) - 1}")

### Q11: If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed?

**Column Index:** 15

In [ ]:
# Q11: Extract and clean responses
docs_q11 = df.iloc[:, 15].dropna().astype(str).str.strip()
docs_q11 = docs_q11[docs_q11.ne("")].tolist()

print(f"Q11 - Total responses: {len(docs_q11)}")

# Fit BERTopic
model_q11 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)

topics_q11, probs_q11 = model_q11.fit_transform(docs_q11)
print(f"Q11 - Topics found: {len(set(topics_q11)) - 1}")

## Store Per-Question Artifacts

Create a dictionary to organize all models, topics, probabilities, and document sets for downstream analysis and tuning.

In [ ]:
# Organize results for easy access and tuning
results = {
    'q1': {'model': model_q1, 'topics': topics_q1, 'probs': probs_q1, 'docs': docs_q1},
    'q2': {'model': model_q2, 'topics': topics_q2, 'probs': probs_q2, 'docs': docs_q2},
    'q3': {'model': model_q3, 'topics': topics_q3, 'probs': probs_q3, 'docs': docs_q3},
    'q4': {'model': model_q4, 'topics': topics_q4, 'probs': probs_q4, 'docs': docs_q4},
    'q5': {'model': model_q5, 'topics': topics_q5, 'probs': probs_q5, 'docs': docs_q5},
    'q6': {'model': model_q6, 'topics': topics_q6, 'probs': probs_q6, 'docs': docs_q6},
    'q7': {'model': model_q7, 'topics': topics_q7, 'probs': probs_q7, 'docs': docs_q7},
    'q8': {'model': model_q8, 'topics': topics_q8, 'probs': probs_q8, 'docs': docs_q8},
    'q9': {'model': model_q9, 'topics': topics_q9, 'probs': probs_q9, 'docs': docs_q9},
    'q10': {'model': model_q10, 'topics': topics_q10, 'probs': probs_q10, 'docs': docs_q10},
    'q11': {'model': model_q11, 'topics': topics_q11, 'probs': probs_q11, 'docs': docs_q11},
}

print("All models fitted and stored in 'results' dictionary.")
print("\nAccess individual results like: results['q1']['model'].get_topic_info()")

## Next Steps: Tuning Individual Questions

To tune parameters for a specific question:

1. Modify the baseline parameters in the **Baseline BERTopic Parameters** section
2. Or override parameters in an individual question cell (create a local `umap_model`, `hdbscan_model`, etc.)
3. Re-run the question cell to refit the model
4. Access visualizations via `results['qN']['model'].visualize_topics()`, etc.
5. Check topic info with `results['qN']['model'].get_topic_info()`

Example tuning workflow for Q1:
```python
# Override UMAP for Q1
umap_q1 = UMAP(n_neighbors=10, n_components=5, min_dist=0.1, metric="cosine", random_state=42)
model_q1 = BERTopic(..., umap_model=umap_q1, ...)
topics_q1, probs_q1 = model_q1.fit_transform(docs_q1)
results['q1'] = {'model': model_q1, 'topics': topics_q1, 'probs': probs_q1, 'docs': docs_q1}
```